# 📘 평균값의 차이 검정

두 그룹 간의 평균에 유의미한 차이가 있는지 검정하는 방법을 배웁니다.

**두 가지 상황:**
- **대응이 있는 t검정**(Paired t-test): 같은 대상의 전후 비교 (예: 약 복용 전후 체온)
- **대응이 없는 t검정**(Independent t-test): 서로 다른 두 그룹 비교 (예: A반 vs B반 성적)

**학습 목표:**
- 대응이 있는/없는 t검정의 차이 이해
- `stats.ttest_rel()`, `stats.ttest_ind()` 사용법
- 검정 결과의 해석

## 1. 데이터 불러오기

약을 복용하기 전후의 체온 데이터를 불러옵니다.
같은 사람(A~E)의 전후 체온이므로 **대응이 있는 데이터**입니다.

In [ ]:
# ┌─────────────────────────────────────────┐
# │  라이브러리 임포트 + 데이터 로드            │
# └─────────────────────────────────────────┘

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

sns.set()
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# 대응이 있는 체온 데이터
paired_data = pd.read_csv("paired_t_test.csv")
print("=== 대응이 있는 데이터 (약 복용 전후 체온) ===")
print(paired_data)

# 전후 데이터 분리
before = paired_data.query('medicine == "before"')["body_temperature"].values
after = paired_data.query('medicine == "after"')["body_temperature"].values
print(f"\n복용 전 체온: {before}")
print(f"복용 후 체온: {after}")
print(f"\n복용 전 평균: {np.mean(before):.2f}°C")
print(f"복용 후 평균: {np.mean(after):.2f}°C")

## 2. 대응이 있는 t검정 (Paired t-test)

**대응이 있는 데이터**는 같은 대상을 두 번 측정한 경우입니다.
예: 약 복용 전후 체온, 다이어트 전후 몸무게

**검정 방법:**
1. 각 대상의 차이(d)를 계산: d = after - before
2. 차이의 평균이 0과 유의미하게 다른지 검정
3. `stats.ttest_rel()` 사용

> 💡 대응이 있는 검정은 개인차를 제거하므로 더 강력합니다.

In [ ]:
# ┌─────────────────────────────────────────┐
# │  대응이 있는 t검정                       │
# │  1) 차이(d) 계산 후 1-sample t검정       │
# │  2) stats.ttest_rel() 직접 사용          │
# └─────────────────────────────────────────┘

# 차이 계산: d = after - before
diff = after - before
print("=== 차이(d = after - before) ===")
print(f"차이: {diff}")
print(f"차이의 평균: {np.mean(diff):.4f}")
print(f"차이의 표준편차: {np.std(diff, ddof=1):.4f}")

# 방법 1: 차이의 평균이 0과 다른지 검정 (1-sample t검정)
result_1samp = stats.ttest_1samp(diff, 0)
print(f"\n=== 1-sample t검정 (차이가 0인지) ===")
print(f"t값: {result_1samp.statistic:.4f}")
print(f"p값: {result_1samp.pvalue:.6f}")

# 방법 2: 대응이 있는 t검정 (결과는 동일)
result_rel = stats.ttest_rel(after, before)
print(f"\n=== stats.ttest_rel() 결과 ===")
print(f"t값: {result_rel.statistic:.4f}")
print(f"p값: {result_rel.pvalue:.6f}")

# 판정
alpha = 0.05
if result_rel.pvalue < alpha:
    print(f"\np값 {result_rel.pvalue:.6f} < {alpha} → 귀무가설 기각!")
    print("→ 약 복용 전후 체온에 유의미한 차이가 있습니다")
else:
    print(f"\np값 {result_rel.pvalue:.6f} >= {alpha} → 귀무가설 채택")
    print("→ 약 복용 전후 체온에 유의미한 차이가 없습니다")

In [ ]:
# ┌─────────────────────────────────────────┐
# │  대응이 있는 데이터 시각화                │
# │  전후 비교 그래프                         │
# └─────────────────────────────────────────┘

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 왼쪽: 전후 체온 비교
x = np.arange(len(before))
axes[0].plot([0]*len(before), before, 'o', color='steelblue', label='복용 전')
axes[0].plot([1]*len(after), after, 'o', color='orange', label='복용 후')
for i in range(len(before)):
    axes[0].plot([0, 1], [before[i], after[i]], 'k-', alpha=0.3)
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(['복용 전', '복용 후'])
axes[0].set_ylabel('체온 (°C)')
axes[0].set_title('약 복용 전후 체온 변화')
axes[0].legend()

# 오른쪽: 차이의 분포
axes[1].bar(range(len(diff)), diff, color=['steelblue' if d > 0 else 'salmon' for d in diff])
axes[1].axhline(0, color='black', linewidth=0.5)
axes[1].set_xlabel('환자 번호')
axes[1].set_ylabel('체온 차이 (°C)')
axes[1].set_title(f'체온 차이 (평균={np.mean(diff):.2f})')

plt.tight_layout()
plt.savefig('paired_ttest.png', dpi=100)
plt.show()

## 3. 대응이 없는 t검정 (Independent t-test)

**대응이 없는 데이터**는 서로 다른 두 그룹을 비교하는 경우입니다.
예: A반과 B반의 시험 점수 비교, 남녀 키 비교

**검정 방법:**
- 두 그룹의 평균 차이가 우연인지 검정
- `stats.ttest_ind()` 사용
- `equal_var=False`로 Welch's t검정 수행 가능

> 💡 대응이 없는 검정은 두 그룹의 분산이 다를 수 있으므로
> Welch's t검정(`equal_var=False`)이 더 안전합니다.

In [ ]:
# ┌─────────────────────────────────────────┐
# │  대응이 없는 t검정                       │
# │  두 그룹의 평균 차이가 유의미한지 검정    │
# │  분산이 다를 수 있으므로 Welch's t검정    │
# └─────────────────────────────────────────┘

# 복용 전후를 서로 다른 그룹으로 취급 (대응이 없다고 가정)
print("=== 대응이 없는 t검정 ===")
print(f"복용 전 평균: {np.mean(before):.4f}")
print(f"복용 후 평균: {np.mean(after):.4f}")
print(f"평균 차이: {np.mean(after) - np.mean(before):.4f}")

# 분산 비교
print(f"\n복용 전 분산: {np.var(before, ddof=1):.4f}")
print(f"복용 후 분산: {np.var(after, ddof=1):.4f}")

# Welch's t검정 (분산이 다를 수 있음)
result_ind = stats.ttest_ind(after, before, equal_var=False)
print(f"\n=== stats.ttest_ind(equal_var=False) ===")
print(f"t값: {result_ind.statistic:.4f}")
print(f"p값: {result_ind.pvalue:.6f}")

# 등분산 가정 t검정 (참고)
result_equal = stats.ttest_ind(after, before, equal_var=True)
print(f"\n=== stats.ttest_ind(equal_var=True) ===")
print(f"t값: {result_equal.statistic:.4f}")
print(f"p값: {result_equal.pvalue:.6f}")

# 판정
alpha = 0.05
if result_ind.pvalue < alpha:
    print(f"\np값 {result_ind.pvalue:.6f} < {alpha} → 귀무가설 기각!")
    print("→ 두 그룹의 평균에 유의미한 차이가 있습니다")
else:
    print(f"\np값 {result_ind.pvalue:.6f} >= {alpha} → 귀무가설 채택")
    print("→ 두 그룹의 평균에 유의미한 차이가 없습니다")

In [ ]:
# ┌─────────────────────────────────────────┐
# │  대응이 있는 vs 없는 t검정 비교           │
# │  대응이 있는 검정이 더 강력함             │
# └─────────────────────────────────────────┘

print("=== 두 검정 방법 비교 ===")
print(f"\n{'검정 방법':<25s} {'t값':>10s} {'p값':>12s}")
print("-" * 50)
print(f"{'대응이 있는 t검정':<25s} {result_rel.statistic:>10.4f} {result_rel.pvalue:>12.6f}")
print(f"{'대응이 없는 t검정(Welch)':<25s} {result_ind.statistic:>10.4f} {result_ind.pvalue:>12.6f}")

print(f"\n💡 대응이 있는 t검정의 p값이 더 작습니다.")
print(f"   개인차를 제거하면 더 정밀한 검정이 가능합니다.")
print(f"   데이터에 대응 관계가 있으면 반드시 대응이 있는 검정을 사용하세요!")

## 📋 평균값 차이 검정 요약

| 검정 종류 | 상황 | 함수 | 특징 |
|-----------|------|------|------|
| 대응이 있는 t검정 | 같은 대상 전후 비교 | `ttest_rel()` | 개인차 제거, 더 강력 |
| 대응이 없는 t검정 | 서로 다른 두 그룹 | `ttest_ind()` | 그룹 간 비교 |
| Welch's t검정 | 분산이 다른 두 그룹 | `ttest_ind(equal_var=False)` | 등분산 가정 불필요 |

> 💡 **선택 기준**: 같은 사람의 전후 데이터 → `ttest_rel()`, 다른 그룹 비교 → `ttest_ind(equal_var=False)`

## 🎯 연습 문제

1. 위 데이터에서 `equal_var=True`와 `equal_var=False`의 결과를 비교하고, 차이를 설명하세요.
2. 새로운 데이터에서 대응이 있는 t검정과 대응이 없는 t검정을 모두 수행하고, p값의 차이를 비교하세요.
3. 유의수준을 0.01로 변경했을 때 검정 결과가 어떻게 달라지는지 확인하세요.
4. 두 그룹의 표본 크기를 2배로 늘렸을 때 p값이 어떻게 변하는지 시뮬레이션하세요.